In [1]:
from promenade.models import *
from promenade.agents import build_graph
from langchain.tools import tool
from langchain.agents import create_agent

In [2]:
model = llm
reranker = RetreiveReranker(
    rerank_n=1000, 
    retrieve_n=10000, 
    rerank_model=RERANKER_MODEL)
WEB_TOOLS = WebTools(READER_URL)
DEV = True
SAMPLES_PATH = PROJECT_ROOT / "docs" / "samples"
base_url = "https://kosmo-museum.ru/" 
graph = build_graph(WEB_TOOLS, reranker, model, DEV=DEV, SAMPLES_PATH=SAMPLES_PATH)

In [3]:
import json

@tool
def parse_and_insert_into_db(url: HttpUrl) -> str:
    """Parse web page and insert into DB.

    Args:
        url (HttpUrl): URL address of the page to parse.

    Returns:
        str: JSON string with keys:
            - status: "success" or "error"
            - message: human-readable description
            - details: additional context
    """
    try:
        state = asyncio.run(graph.ainvoke({"url": base_url, "subdocs": [], "results": []}))
        count  = sum(1 for r in state["results"] if r["ok"])
        result: ToolResult = {
            "status": "success", 
            "message": f"Sucssefully created {count} entities in the sql database and vesctor database.", 
            "details": {
                "count": count, 
                "result": state["results"]
            }
        }
        return json.dumps(result, ensure_ascii=False)

    except Exception as e:
        result: ToolResult = {
            "status": "error", 
            "message": str(e), 
            "details": {}
        }
        return json.dumps(result, ensure_ascii=False)

In [ ]:
GENERAL_AGENT_SYSTEM_PROMPT = """
You are Promenade — an AI assistant for discovering cultural leisure activities in Moscow.

## Core Functionality
Promenade crawls venue pages (museums, exhibitions, concerts, festivals), extracts structured information,
persists data to SQL database and Qdrant vector store, and answers natural-language queries through retrieval-augmented generation.

## Available Modes (Agent Modes)
The agent supports multiple operating modes, selected based on user input or context:

### 1. Crawl & Store Mode (current)
When the user provides a URL to a venue page:
- Use `parse_and_insert_into_db` tool to crawl and store venue information
- Extracts schedule, ticket prices, exhibitions, and contact details
- Stores structured data in SQL database (`museum`, `schedule` tables)
- Creates vector embeddings in Qdrant for semantic search
- After getting a result of the tool only answer the user about quantity of entities created in database and their names.
- Do not suggest further interaction with user

### 2. Query & Retrieve Mode (future)
When the user asks natural-language questions about venues:
- Use retrieval from Qdrant to find relevant venues
- Rerank results by relevance to the query
- Generate answers using LLM with retrieved context

### 3. Search Mode (future)
When the user wants to browse venues by category:
- Filter venues by type (museum, concert, festival, etc.)
- Filter by schedule (open now, weekend, specific date)
- Filter by price range or special offers

## Guidelines
- In Crawl & Store mode, only use the provided URL (do not search for others)
- In Query mode, always cite sources from the retrieved data
- When uncertain, ask clarifying questions rather than making assumptions

## Output Format
- Use Russian for all responses unless user specifies otherwise
"""

agent = create_agent(
    model=model,
    tools=[parse_and_insert_into_db],
    system_prompt=GENERAL_AGENT_SYSTEM_PROMPT,
)

In [5]:
agent_result = agent.invoke(
    {"messages": [{"role": "user", "content": "Хочу как-нибудь сходить в https://kosmo-museum.ru/"}]},
)
print(agent_result)

Processed: 2/2
{'messages': [HumanMessage(content='Хочу как-нибудь сходить в https://kosmo-museum.ru/', additional_kwargs={}, response_metadata={}, id='1a1ecade-7692-4c96-b046-90cd452d31c2'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 624, 'total_tokens': 654, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 192}}, 'model_provider': 'openai', 'model_name': 'Qwen/Qwen3-235B-A22B-Instruct-2507', 'system_fingerprint': None, 'id': 'chatcmpl-0bc77a69-eabb-40f6-801b-ef3067022576', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019de807-585d-7382-a8e6-1eab780ef8b7-0', tool_calls=[{'name': 'parse_and_insert_into_db', 'args': {'url': 'https://kosmo-museum.ru/'}, 'id': 'chatcmpl-tool-2624f877d11a4db9a0a696ec5c42ce9e', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 624, 'output_tokens': 30, 'total_tokens': 654

In [6]:
print(agent_result['messages'][-2].content)
print(agent_result['messages'][-1].content)

{"status": "success", "message": "Sucssefully created 2 entities in the sql database and vesctor database.", "details": {"count": 2, "result": [{"ok": [true, null], "doc_preview": "Base url: https://kosmo-museum.ru/ \n\n## Музей косм", "place_name": "Музей космонавтики"}, {"ok": [true, null], "doc_preview": "Base url: https://kosmo-museum.ru/ \n\n## Мемориальн", "place_name": "Мемориальный дом-музей академика С.П. Королёва"}]}}
Успешно добавлено 2 объекта в базу данных:  
- Музей космонавтики  
- Мемориальный дом-музей академика С.П. Королёва
